In [ ]:
# importing the libraries
import pandas as pd 
import numpy as np 
from pathlib import Path
import re
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
# setting the path and loading the dataset

DATA_DIR    = Path("Data download/data")
INPUT_PATH  = DATA_DIR / "filings_clean.csv"
LM_PATH = DATA_DIR / "Loughran-McDonald_MasterDictionary_1993-2025.csv"
LM_CHECKPOINT_PATH = DATA_DIR / "lm_checkpoint.csv"

df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} filings")
print(f"Columns: {list(df.columns)}")

In [ ]:
lm = pd.read_csv(LM_PATH)
print(f"Shape: {lm.shape}")
print(f"Columns: {list(lm.columns)}")

In [ ]:
# ── Loughran-McDonald Dictionary Scorer ──────────────────────────────────────

lm_negative = set(lm[lm['Negative'] != 0]['Word'].str.upper())
lm_positive = set(lm[lm['Positive'] != 0]['Word'].str.upper())

print(f"LM negative words : {len(lm_negative)}")
print(f"LM positive words : {len(lm_positive)}")

def score_lm(text: str) -> dict:
    if not isinstance(text, str):
        return {'lm_neg_prop': 0.0, 'lm_net': 0.0, 'lm_n_tokens': 0}

    tokens = re.findall(r'[a-zA-Z]+', text.upper())
    total = len(tokens)
    if total == 0:
        return {'lm_neg_prop': 0.0, 'lm_net': 0.0, 'lm_n_tokens': 0}

    neg_count = sum(1 for t in tokens if t in lm_negative)
    pos_count = sum(1 for t in tokens if t in lm_positive)

    return {
        'lm_neg_prop': neg_count / total,
        'lm_net':      (neg_count - pos_count) / total,
        'lm_n_tokens': total,
    }


# Test on a few examples
test_cases = [
    "Revenue increased 15% and earnings exceeded expectations strong growth outlook",
    "Revenue declined significantly loss widened restructuring charges uncertainty risk",
    "Results were in line with expectations no change to guidance",
]

for test in test_cases:
    s = score_lm(test)
    print(f"neg_prop={s['lm_neg_prop']:.4f}  net={s['lm_net']:.4f} | {test[:60]}")

In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas()

scores = df['cleanText'].progress_apply(score_lm)
df = df.join(pd.DataFrame(scores.tolist()))

print("LM scoring complete.")
print("\nScore distribution:")
print(df[['lm_neg_prop', 'lm_net', 'lm_n_tokens']].describe().round(4))
print(f"\nMean lm_neg_prop  downside=1 : {df[df['downside']==1]['lm_neg_prop'].mean():.4f}")
print(f"Mean lm_neg_prop  downside=0 : {df[df['downside']==0]['lm_neg_prop'].mean():.4f}")
print(f"Mean lm_net       downside=1 : {df[df['downside']==1]['lm_net'].mean():.4f}")
print(f"Mean lm_net       downside=0 : {df[df['downside']==0]['lm_net'].mean():.4f}")

In [ ]:
auc_neg_prop = roc_auc_score(df['downside'], df['lm_neg_prop'])
auc_net      = roc_auc_score(df['downside'], df['lm_net'])
print(f"LM Neg Proportion AUC : {auc_neg_prop:.4f}  (standard LM score)")
print(f"LM Net Sentiment AUC  : {auc_net:.4f}  (neg - pos) / total")

decile_threshold = df['lm_neg_prop'].quantile(0.9)
bottom_decile = df[df['lm_neg_prop'] >= decile_threshold]
print(f"\nBottom decile avg CAR : {bottom_decile['car_0_1'].mean():.4f}")
print(f"Bottom decile n       : {len(bottom_decile)}")

In [ ]:
df[['accessionNumber', 'ticker', 'filingDate', 'car_0_1', 'downside',
    'lm_neg_prop', 'lm_net', 'lm_n_tokens']].to_csv(LM_CHECKPOINT_PATH, index=False)

print(f"Saved to {LM_CHECKPOINT_PATH}")
print(f"Rows: {len(df)}")

The Loughran-McDonald dictionary produces near-random rankings across both scoring variants. The negative proportion score achieves AUC of 0.5002 and the net sentiment score achieves 0.5078, both statistically indistinguishable from the o.5 baseline of a random classifier. With 5284 observations, the standard error under the null is approximately 0.007, placing both values well within the range of chance variation. 

The bottom decile result reinforces this finding. The 529 filings flagged as most negative by the dictionary have an average CAR of -0.38%, which is economically trivial and barely distinguishable from the full sample mean. In other words, the filings the dictionary identifies as most negative do not correspond to filings that actually moved markets downward.

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.32)

# ── Plot 1: Score distribution by downside label ──────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df[df['downside']==0]['lm_neg_prop'], bins=60, alpha=0.6, 
         color='steelblue', label='No downside', density=True)
ax1.hist(df[df['downside']==1]['lm_neg_prop'], bins=60, alpha=0.6, 
         color='tomato', label='Downside', density=True)
ax1.set_xlabel('LM Negative Proportion')
ax1.set_ylabel('Density')
ax1.set_title('Score Distribution by Downside Label')
ax1.legend(frameon=False)

# ── Plot 2: ROC curve ─────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
fpr, tpr, _ = roc_curve(df['downside'], df['lm_neg_prop'])
auc = roc_auc_score(df['downside'], df['lm_neg_prop'])
ax2.plot(fpr, tpr, color='steelblue', lw=1.5, label=f'LM Neg Prop (AUC = {auc:.4f})')
ax2.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.5, label='Random (AUC = 0.50)')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve — LM Dictionary')
ax2.legend(frameon=False, fontsize=9)

# ── Plot 3: Average CAR by score decile ───────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
df['lm_decile'] = pd.qcut(df['lm_neg_prop'], q=10, labels=False) + 1
decile_car = df.groupby('lm_decile')['car_0_1'].mean()
colors = ['tomato' if i == 10 else 'steelblue' for i in decile_car.index]
ax3.bar(decile_car.index, decile_car.values * 100, color=colors, edgecolor='white', linewidth=0.5)
ax3.axhline(0, color='black', lw=0.8)
ax3.set_xlabel('LM Score Decile (1 = least negative, 10 = most negative)')
ax3.set_ylabel('Mean CAR[0,+1] (%)')
ax3.set_title('Average CAR by LM Score Decile')
ax3.set_xticks(range(1, 11))

# ── Plot 4: Downside rate by score decile ─────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
decile_rate = df.groupby('lm_decile')['downside'].mean() * 100
baseline = df['downside'].mean() * 100
ax4.bar(decile_rate.index, decile_rate.values, color='steelblue', edgecolor='white', linewidth=0.5)
ax4.axhline(baseline, color='tomato', lw=1.2, linestyle='--', label=f'Sample baseline ({baseline:.1f}%)')
ax4.set_xlabel('LM Score Decile (1 = least negative, 10 = most negative)')
ax4.set_ylabel('Downside Rate (%)')
ax4.set_title('Downside Rate by LM Score Decile')
ax4.set_xticks(range(1, 11))
ax4.legend(frameon=False, fontsize=9)

plt.suptitle('Loughran-McDonald Dictionary — Evaluation', fontsize=13, fontweight='500', y=1.01)
plt.show()

Plot 1 (Score Distribution): The near-identical distributions of downside and non-downside filings confirm that the LM dictionary assigns similar negative word proportions to both groups, leaving no meaningful separation.
Plot 2 (ROC Curve): The ROC curve traces almost exactly along the diagonal random baseline, with an AUC of 0.5002 confirming the dictionary has no discriminative power over this sample.
Plot 3 (Average CAR by Decile): Despite decile 10 showing the most negative average CAR, the pattern across deciles is irregular and non-monotonic, indicating the relationship is driven by noise rather than a systematic signal.
Plot 4 (Downside Rate by Decile): The downside rate fluctuates randomly around the 24.7% sample baseline across all deciles, with no tendency for higher-scoring filings to be associated with greater downside frequency.

In [ ]:
print("=== LM Score Separation ===")
for col in ['lm_neg_prop', 'lm_net']:
    d1 = df[df['downside']==1][col].mean()
    d0 = df[df['downside']==0][col].mean()
    diff = d1 - d0
    print(f"{col}:")
    print(f"  downside=1 : {d1:.6f}")
    print(f"  downside=0 : {d0:.6f}")
    print(f"  difference : {diff:.6f}")
    print()

# Also check token count — are we scoring enough text?
print(f"Median tokens scored : {df['lm_n_tokens'].median():.0f}")
print(f"Mean tokens scored   : {df['lm_n_tokens'].mean():.0f}")
print(f"Min tokens scored    : {df['lm_n_tokens'].min():.0f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load the GPT checkpoint to get the exact same 2000 accession numbers
gpt_df = pd.read_csv(DATA_DIR / "gpt_checkpoint.csv")

# Filter LM scores to those same 2000 filings
lm_sub = df[df['accessionNumber'].isin(gpt_df['accessionNumber'])].copy()
print(f"LM subsample      : {len(lm_sub)} filings")
print(f"Downside          : {lm_sub['downside'].sum()}")
print(f"Non-downside      : {(lm_sub['downside']==0).sum()}")

# Apply same 80/20 split with same random seed as GPT
val_df, test_df = train_test_split(
    lm_sub,
    test_size=0.80,
    stratify=lm_sub['downside'],
    random_state=42
)

print(f"\nValidation set    : {len(val_df)} filings")
print(f"Test set          : {len(test_df)} filings")

# Full sample AUC
auc_full = roc_auc_score(df['downside'], df['lm_neg_prop'])
print(f"\nLM AUC (full, n=5280)       : {auc_full:.4f}")

# Subsample AUC
auc_sub = roc_auc_score(lm_sub['downside'], lm_sub['lm_neg_prop'])
print(f"LM AUC (subsample, n=2000)  : {auc_sub:.4f}")

# Test set AUC
auc_test = roc_auc_score(test_df['downside'], test_df['lm_neg_prop'])
print(f"LM AUC (test set, n=1600)   : {auc_test:.4f}")

# Bootstrap CI on test set
np.random.seed(42)
boot_aucs = []
for _ in range(1000):
    sample = test_df.sample(len(test_df), replace=True)
    try:
        boot_aucs.append(roc_auc_score(sample['downside'], sample['lm_neg_prop']))
    except:
        pass

ci_low  = np.percentile(boot_aucs, 2.5)
ci_high = np.percentile(boot_aucs, 97.5)
print(f"95% bootstrap CI            : [{ci_low:.4f}, {ci_high:.4f}]")

# Save test set accession numbers for FinBERT to use the same ones
test_df[['accessionNumber']].to_csv(DATA_DIR / "test_accessions.csv", index=False)
print(f"\nTest accessions saved for FinBERT comparison.")